# 基础Prompt

prompt是大模型的核心,传统方式一般是使用字符串凭借或者模板字符串实现,而使用langChain可以构建可复用的prompy来让我们更工程化的管理prompt 从而制作复杂的Chatbot

首先是基础的`prompt template` 帮助我们快速定义一个包含变量的字符串模板, 我们可以用过向该类的对象输入不同的变量值来动态生成不同的prompt 这可以方便的让我们定义一组prompt模板 然后在运行时根据用户的动态输入来生成prompt


# 无变量的template

`PromptTemplate` 就是最基础的 `template`，我们不传入任何变量（`inputVariables: []`），这跟硬编码一个字符串没任何区别。 调用 prompt template 的方式就是 format，因为我们没有任何变量，也就没有任何参数。

没有变量的 prompt template 使用的很少，这里主要以此帮助大家理解 template 的概念

In [1]:
import { PromptTemplate } from "@langchain/core/prompts";

const greetingPrompt = new PromptTemplate({
  inputVariables: [],
  template: "hello world",
});

const formattedGreetingPrompt = await greetingPrompt.format();

console.log(formattedGreetingPrompt);
// hello world


hello world


# 含变量的template

其API 使用`{}`来包裹变量,然后在 `inputVariables` 声明用到的变量名称。因为有了变量，所以在调用 `format()` 就需要传入对应的变量。

In [2]:
const personalizaedGreetingPrompt = new PromptTemplate({
  inputVariables:["name"],
  template:'hello {name}'
});
const formattedPersonalizaedGreeting = await personalizaedGreetingPrompt.format({
  name:"Kai"
});
console.log(formattedPersonalizaedGreeting);
// hello Kai


hello Kai


对于多变量也可以这么调用

In [3]:
const multiVariableGreetingPrompt = new PromptTemplate({
  inputVariables: ["timeOfDay", "name"],
  template: "good {timeOfDay}, {name}",
});
const formattedMultiVariableGreeting = await multiVariableGreetingPrompt.format({
  timeOfDay: "morning",
  name: "Kai",
});

console.log(formattedMultiVariableGreeting);
// good morning, Kai


good morning, Kai


唯一需要注意的就是，如果你的 prompt 需要 {}，可以这么转义{{}}

In [4]:
const multiVariableGreetingPrompts = new PromptTemplate({
  inputVariables:["timeOfDay","name"],
  template:"good {timeOfDay}, {name},{{test}}",
});

这样创建 prompt 的时候，会自动从字符串中推测出需要输入的变量。

# 使用部分参数创建 template

我们并不需要一次性把所有变量都输入进去，在工程中，我们可能先获得某个参数，之后才能获得另一个参数。这里类似于函数式编程的概念，我们给 需要两个参数的 prompt template 传递一个参数后，就会生成需要一个参数的 prompt template。

使用`partial`方法获取第一个参数,然后生成剩余参数的模板,当然也可以直接使用`format`方法获取两个模板

In [5]:
const initialPrompt = new PromptTemplate({
  inputVariables:['type','item'],
  template:"这是一个{type},他是{item}"
})

const typePrompt= await initialPrompt.partial({
  type:'工具'
})

const itemPrompt = await typePrompt.format({
  item:"锤子"
})

console.log(itemPrompt);


这是一个工具,他是锤子


# 使用动态填充参数

当我们需要，一个 prompt template 被 format 时，实时地动态生成参数时，我们可以使用函数来对 template 部分参数进行指定。


In [6]:
const getCurrentDateStr = () => {
  return new Date().toLocaleDateString();
};

const promptWithDate = new PromptTemplate({
  template: "今天是{date},{active}",
  inputVariables: ["date", "active"],
});

const partialPrompt =await promptWithDate.partial({
  date:getCurrentDateStr()
});

const formattedPrompt = await partialPrompt.format({
  active:"活动"
});

console.log(formattedPrompt);


今天是2026/4/7,活动


注意，函数 `getCurrentDateStr` 是在 `format` 被调用的时候实时运行的，也就是可以在被渲染成字符串时获取到最新的外部信息。 目前这里不支持传入参数，如果需要参数，可以用 js 的闭包进行参数的传递。
假设我们有一个根据时间段（morning, afternoon, evening）返回不同问候语，并且需要带上当前时间的需求

In [7]:
const getCurrentDateStrs = () => {
  return new Date().toLocaleDateString();
};

function generateGreeting(timeOfDay: string) {
  return () => {
    const date = getCurrentDateStrs()
    switch (timeOfDay) {
      case 'morning':
        return date + ' 早上好';
      case 'afternoon':
        return date + ' 下午好';
      case 'evening':
        return date + ' 晚上好';
      default:
        return date + ' 你好';
    }
  };
}

const prompts = new PromptTemplate({
  template: "{greeting}!",
  inputVariables: ["greeting"],
});

const currentTimeOfDay = 'afternoon';
const partialPrompts = await prompts.partial({
  greeting: generateGreeting(currentTimeOfDay),
});

const formattedPrompts = await partialPrompts.format();

console.log(formattedPrompts);
// 输出: 3/21/2024 下午好!


2026/4/7 下午好!


# Chat Prompt

目前Chat Api是和LLM交流的主要形式,所以`ChatPromptTemplate`也是目前最常用的工具.

跟各种聊天模型交互的时候，在构建聊天信息时，不仅仅包含了像上文中的文本内容，也需要与每条消息关联的角色信息。 例如这条信息是由 人类、AI、还是给 chatbot 指定的 system 信息，这种结构化的消息输入有助于模型更好地理解对话的上下文和流程，从而生成更准确、更自然的回应。

为了方便地构建和处理这种结构化的聊天消息，LangChain 提供了几种与聊天相关的提示模板类，如 `ChatPromptTemplate、SystemMessagePromptTemplate、AIMessagePromptTemplate` 和 `HumanMessagePromptTemplate。`

其中这些模板提示词都有分配一个角色:

- `system` : 通过海沧用于设置对话的上下文规则,这些消息不会直接显示在对话中,但是对于模型的行为具有指导意义. 可以理解为模型的初始信息, 权重非常高.
- `user`: 用户在对话中的发言,这些消息直接显示在对话中,反应用户需求
- `assistant`: AI模型的回复,根据system和user的消息,生成模型的回复.

下面我们以一个基础的翻译 chatbot 来讲解这几个常见 chat template，我们先构建一个 system message 来给 llm 指定核心的准则:

In [8]:
import {SystemMessagePromptTemplate} from "@langchain/core/prompts";

const translateInstruction = SystemMessagePromptTemplate.fromTemplate(`你是一个专业的翻译人员,你的任务是将文本从{source_Language}翻译成{target_Language}`)

构建一个用户输入的信息

In [9]:
import {HumanMessagePromptTemplate} from "@langchain/core/prompts"

const userQuestionTemplate = HumanMessagePromptTemplate.fromTemplate(`请翻译这句话：{text}`)

然后将这两个组合起来形成一个对话信息

In [10]:
import {ChatPromptTemplate} from "@langchain/core/prompts"

const chatPrompt = ChatPromptTemplate.fromMessages([translateInstruction, userQuestionTemplate])


然后使用一个`fromMessages`格式化整个对话

In [11]:
const formattedChatPrompt=await chatPrompt.formatMessages({
  source_Language:"English",
  target_Language:"Chinese",
  text:"Hello, I am a student.",
})

console.log(formattedChatPrompt);



[
  SystemMessage {
    lc_serializable: true,
    lc_kwargs: {
      content: "你是一个专业的翻译人员,你的任务是将文本从English翻译成Chinese",
      additional_kwargs: {},
      response_metadata: {}
    },
    lc_namespace: [ "langchain_core", "messages" ],
    content: "你是一个专业的翻译人员,你的任务是将文本从English翻译成Chinese",
    name: undefined,
    additional_kwargs: {},
    response_metadata: {}
  },
  HumanMessage {
    lc_serializable: true,
    lc_kwargs: {
      content: "请翻译这句话：Hello, I am a student.",
      additional_kwargs: {},
      response_metadata: {}
    },
    lc_namespace: [ "langchain_core", "messages" ],
    content: "请翻译这句话：Hello, I am a student.",
    name: undefined,
    additional_kwargs: {},
    response_metadata: {}
  }
]


构建了一个数组，每一个元素都是一个 Message。 同样的 chatPrompt 也有简便写法的语法糖

简化语法如下:
```ts
const systemTemplate = "你是一个专业的翻译员，你的任务是将文本从{source_lang}翻译成{target_lang}。";
const humanTemplate = "请翻译这句话：{text}";

const chatPrompt = ChatPromptTemplate.fromMessages([
  ["system", systemTemplate],
  ["human", humanTemplate],
]);
```

我们穿件一个Chain测试一下

In [12]:
import { load } from "dotenv";
await load({ 
export: true,
envPath:'../.env'
});

{ OPENAI_API_KEY: "sk-be17cc28883b4b76b8924f23e4ca522b" }

In [13]:
import { ChatOpenAI } from "@langchain/openai"
import { StringOutputParser } from "@langchain/core/output_parsers"

const chatModel = new ChatOpenAI({
  modelName:"deepseek-chat",
  configuration: {
    baseURL: "https://api.deepseek.com",
  }
})

const outputParsers= new StringOutputParser();
const chain = chatPrompt.pipe(chatModel).pipe(outputParsers)

await chain.invoke({
  source_Language:"en",
  target_Language:"zh",
  text:"Hello,world!",
})


"你好，世界！"

# 组合多个 Prompt

在实际工程中,我们可能会根据多个变量,根据多个外接环境去构造复杂prompt,这就是`PipeLinePromptTemplate`的应用场景 . 可以将多个独立的temple构建为一个完整且复杂的prompt,这样就可以提高独立prompt的复用性.

在 `PipelinePromptTemplate` 有两个核心的概念：

- `pipelinePrompts`，一组 object，每个 object 表示 `prompt` 运行后赋值给 `name` 变量
- `finalPrompt`，表示最终输出的 prompt

In [14]:
import {
  PipelinePromptTemplate,
} from "@langchain/core/prompts";


const fullPrompt = PromptTemplate.fromTemplate(`
你是一个智能管家，今天是 {date}，你的主人的信息是{info}, 
根据上下文，完成主人的需求
{task}`);

const datePrompt = PromptTemplate.fromTemplate("{date}，现在是 {period}")
const periodPrompt = await datePrompt.partial({
    date: getCurrentDateStr
})

const infoPrompt =  PromptTemplate.fromTemplate("姓名是 {name}, 性别是 {gender}");

const taskPrompt = PromptTemplate.fromTemplate(`
我想吃 {period} 的 {food}。 
再重复一遍我的信息 {info}`);

const composedPrompt = new PipelinePromptTemplate({
  pipelinePrompts: [
    {
      name: "date",
      prompt: periodPrompt,
    },
    {
      name: "info",
      prompt: infoPrompt,
    },
    {
      name: "task",
      prompt: taskPrompt,
    },
  ],
  finalPrompt: fullPrompt,
});

const formattedPrompt1 = await composedPrompt.format({
    period: "早上",
    name: "张三",
    gender: "male",
    food: "lemon"
});

console.log(formattedPrompt1)



你是一个智能管家，今天是 2026/4/7，现在是 早上，你的主人的信息是姓名是 张三, 性别是 male, 
根据上下文，完成主人的需求

我想吃 早上 的 lemon。 
再重复一遍我的信息 姓名是 张三, 性别是 male


这里有几个需要注意的地方

- 一个变量可以多次复用，例如外界输入的 `period` 在 `periodPrompt` 和 `taskPrompt` 都被使用了
- `pipelinePrompts` 中的变量可以被引用，例如我们在 `taskPrompt` 使用了 `infoPrompt` 的运行结果
- 支持动态自定义和 `partial`。例子中我们也涉及到了这两种特殊的 `template`
- langchain 会自动分析 `pipeline` 之间的依赖关系，尽可能的进行并行化来提高运行速度

有了 `pipelinePrompts` 我们可以极大程度的复用和管理我们的 prompt template，从而让 llm app 的开发更加工程化。